# Cat-Dog Image Classifier

### Importing Libraries

In [8]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory  # pyright: ignore
from tensorflow.keras import layers, Sequential # pyright: ignore
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint # pyright: ignore

### Loading Images

In [9]:
train = image_dataset_from_directory(
    '../Data/catdog/training_set',
    validation_split = 0.2,
    subset = 'training',
    seed = 123,
    image_size = (180, 180),
    batch_size = 32
)

validation = image_dataset_from_directory(
    '../Data/catdog/training_set',
    validation_split = 0.2,
    subset = 'validation',
    seed = 123,
    image_size = (180, 180),
    batch_size = 32
)


test = image_dataset_from_directory(
    '../Data/catdog/test_set',
    seed = 123,
    image_size = (180, 180),
    batch_size = 32
)

Found 8000 files belonging to 2 classes.
Using 6400 files for training.
Found 8000 files belonging to 2 classes.
Using 1600 files for validation.
Found 2000 files belonging to 2 classes.


### Scaling and augmenting 

In [10]:
augmenting = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])
rescale = layers.Rescaling(1./255)

### Building The Model

In [11]:
model = Sequential([
    augmenting,
    rescale,
    layers.Conv2D(32,3, activation= 'relu', input_shape = (180,180,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64,3, activation= 'relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128,3, activation= 'relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation= 'relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation= 'sigmoid')
])

### Compiling the model

In [12]:
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

### Training the Model

In [13]:
early_stop = EarlyStopping(monitor= 'val_loss', patience= 5, restore_best_weights= True)
checkpoint = ModelCheckpoint('best_model.keras', monitor= 'val_accuracy', save_best_only= True)

history = model.fit(
    train,
    validation_data  = validation,
    epochs = 40,
    callbacks = [early_stop, checkpoint]
)

Epoch 1/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.5211 - loss: 0.7100 - val_accuracy: 0.5500 - val_loss: 0.6825
Epoch 2/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.5700 - loss: 0.6805 - val_accuracy: 0.5881 - val_loss: 0.6761
Epoch 3/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.5961 - loss: 0.6681 - val_accuracy: 0.6562 - val_loss: 0.6429
Epoch 4/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 34ms/step - accuracy: 0.6425 - loss: 0.6377 - val_accuracy: 0.6100 - val_loss: 0.6589
Epoch 5/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.6711 - loss: 0.6052 - val_accuracy: 0.6938 - val_loss: 0.5847
Epoch 6/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.6922 - loss: 0.5901 - val_accuracy: 0.7038 - val_loss: 0.5685
Epoch 7/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - accuracy: 0.7069 - loss: 0.5697 - val_accuracy: 0.7075 - val_loss: 0.5766
Epoch 8/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - accuracy: 0.7253 - loss: 0.5433 - val_accu

### Testing 

In [14]:
test_loss, test_accuracy = model.evaluate(test)
print(f"Test Accuracy : {test_accuracy}")
print(f"Test loss : {test_loss}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8405 - loss: 0.3772
Test Accuracy : 0.840499997138977
Test loss : 0.37721318006515503
